# Diffusion Background Augmentation

This notebook uses SDXL inpainting to generate new backgrounds for the synthetic
replica images while preserving the original tank pixels. For each image, a fresh
background is generated from a text prompt and the segmented tank is then
composited back on top, so that only the surrounding context changes.

The pipeline runs in three stages: validate the segmentation masks, then generate
backgrounds for the valid masks, repeating across prompt categories until the
target number of augmented images per class is reached. Masks are validated first
because a poor mask would let the generated background bleed onto the vehicle,
which would corrupt the class label in this fine-grained task.

## 1. Setup

Mount Drive, set the base path, and inspect the folder structure before loading
the model.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/augmentation_data')
print(f'Base path: {BASE}')
print(f'Exists: {BASE.exists()}')
if BASE.exists():
    print(f'Contents (top 20): {sorted(os.listdir(BASE))[:20]}')

## 2. Dependencies and model loading

Install the required packages, free up GPU memory, and load the SDXL inpainting
pipeline. CPU offloading and attention/VAE slicing are enabled to keep the model
within the VRAM available on a Colab GPU.

In [ ]:
!pip install -q diffusers transformers accelerate

import gc
import time
import torch
import numpy as np
import cv2
from PIL import Image as PILImage
from diffusers import AutoPipelineForInpainting

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Free memory if a pipeline is already loaded (e.g. when re-running this cell)
if 'pipe' in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

pipe = AutoPipelineForInpainting.from_pretrained(
    'diffusers/stable-diffusion-xl-1.0-inpainting-0.1',
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

print('SDXL loaded successfully')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
else:
    print('No GPU detected')

## 3. Prompt definitions

The background prompts are grouped into five environment categories: agricultural,
forest, roads, urban, and winter. These represent plausible operational settings
while spanning open terrain, vegetation, infrastructure, and seasonal conditions.
The negative prompt suppresses faces, text, and any vehicle- or machinery-like
elements, since these could otherwise be generated into the background and
introduce misleading cues.

In [ ]:
BACKGROUND_PROMPTS = {
    'agricultural': [
        'eastern european wheat field, summer daylight, photorealistic',
        'plowed agricultural field, rural terrain, natural lighting',
        'grassland with dirt patches, eastern europe, daytime',
        'sunflower field with farm tracks, ukrainian countryside',
    ],
    'forest': [
        'birch forest edge, mixed vegetation, eastern europe',
        'forest clearing with tall grass, natural daylight',
        'pine forest with shadows, eastern european wilderness',
    ],
    'roads': [
        'rural dirt road through countryside, eastern europe',
        'muddy track with tire ruts, rural area',
        'cracked asphalt country road',
    ],
    'urban': [
        'abandoned industrial yard, concrete and weeds, eastern europe',
        'rural village outskirts, dirt and grass',
    ],
    'winter': [
        'snow-covered field, overcast winter day, eastern europe',
        'muddy spring terrain with melting snow patches',
    ],
}

NEGATIVE_PROMPT = (
    'faces, text, watermark, blurry, low quality, '
    'cartoon, illustration, painting, anime, 3d render, '
    'vehicles, tanks, military vehicles, cannons, guns, weapons, '
    'deformed, distorted, weird artifacts, '
    'drone, propeller, helicopter, aircraft, rotors, '
    'machine parts, mechanical attachments'
)

# Flatten into (category, prompt) pairs so the batch loop can cycle through them
ALL_PROMPTS = [
    (category, prompt)
    for category, prompts in BACKGROUND_PROMPTS.items()
    for prompt in prompts
]

print(f'Total: {len(ALL_PROMPTS)} prompts across {len(BACKGROUND_PROMPTS)} categories')

## 4. Background generation function

This function generates a new background from a text prompt and composites the
original tank back on top. The tank pixels therefore come entirely from the
original image, while only the surrounding context is generated.

The background is generated at 1024x768 and then resized to the original image
dimensions. The foreground mask is lightly Gaussian-blurred before compositing so
that the boundary between the preserved tank and the generated background is not
a hard edge.

In [ ]:
def inpaint_background(image_path, mask_path, output_path, prompt, seed=None):
    image = PILImage.open(image_path).convert('RGB')
    original_size = image.size

    mask_full = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask_full is None:
        raise ValueError(f'Could not read mask: {mask_path}')
    _, mask_full = cv2.threshold(mask_full, 127, 255, cv2.THRESH_BINARY)

    # Generate the background on a blank canvas at the model's native resolution
    SD_W, SD_H = 1024, 768
    blank = PILImage.new('RGB', (SD_W, SD_H), (128, 128, 128))
    blank_mask = PILImage.new('L', (SD_W, SD_H), 255)

    generator = (torch.Generator(device='cuda').manual_seed(seed)
                 if seed is not None else None)

    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=blank,
        mask_image=blank_mask,
        num_inference_steps=40,
        guidance_scale=9,
        strength=1.0,
        generator=generator,
        height=SD_H,
        width=SD_W,
    ).images[0]

    # Composite the original tank back onto the generated background
    background = result.resize(original_size, PILImage.LANCZOS)
    alpha_mask = cv2.GaussianBlur(mask_full, (3, 3), 0)
    alpha_pil = PILImage.fromarray(alpha_mask).convert('L')
    final = PILImage.composite(image, background, alpha_pil)
    final.save(output_path, quality=95)

print('inpaint_background defined')

## 5. Mask validation

Before generating any backgrounds, the segmentation masks are validated. A usable
mask should cover a plausible fraction of the frame and consist of essentially one
connected region. Masks that are too small, too large, fragmented into many
components, or missing their original image are rejected, so that only reliable
foregrounds enter the generation stage.

In [ ]:
CLASS_NAME = 't90'   # change to 't72' or 't80' to process the other classes


def validate_mask(mask_path, min_coverage=0.05, max_coverage=0.6):
    """Return (is_valid, reason) for a segmentation mask."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return False, 'Cannot read mask'

    _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    coverage = (mask_bin > 0).sum() / mask_bin.size
    if coverage < min_coverage:
        return False, f'Mask too small ({coverage:.1%})'
    if coverage > max_coverage:
        return False, f'Mask too large ({coverage:.1%})'

    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    significant = sum(1 for i in range(1, num_labels)
                      if stats[i, cv2.CC_STAT_AREA] > 100)
    if significant == 0:
        return False, 'No significant components'
    if significant > 3:
        return False, f'Too many components ({significant})'

    return True, f'OK (coverage={coverage:.1%}, components={significant})'


masks_dir = BASE / f'{CLASS_NAME}_masks'
originals_dir = BASE / CLASS_NAME

# Only B1 (neutral background) masks are used for augmentation
all_b1_masks = sorted([f for f in os.listdir(masks_dir)
                       if f.endswith('_mask.png') and '_B1_' in f])

print(f'Validating {len(all_b1_masks)} B1 masks for {CLASS_NAME}...')

valid_masks = []
invalid_masks = []
for mask_file in all_b1_masks:
    basename = mask_file.replace('_mask.png', '')
    if not (originals_dir / f'{basename}.JPG').exists():
        invalid_masks.append((basename, 'missing original'))
        continue

    is_valid, reason = validate_mask(str(masks_dir / mask_file))
    if is_valid:
        valid_masks.append(mask_file)
    else:
        invalid_masks.append((basename, reason))

print(f'\nValid: {len(valid_masks)}, Invalid: {len(invalid_masks)}')
if invalid_masks:
    print('\nInvalid masks (first 20):')
    for name, reason in invalid_masks[:20]:
        print(f'  {name}: {reason}')

## 6. Batch generation

Backgrounds are generated for the validated masks until the target number of
augmented images is reached. The loop cycles through the prompt categories across
successive rounds, so that repeated use of the same mask receives a different
background each time. The run is resumable: existing output files are skipped, so
the loop can be interrupted and restarted without redoing work.

In [ ]:
TARGET_COUNT = 1600
output_dir = BASE / f'{CLASS_NAME}_augmented_B1'
output_dir.mkdir(parents=True, exist_ok=True)


def count_outputs():
    return len([f for f in os.listdir(output_dir) if f.endswith('.jpg')])


print(f'Processing {len(valid_masks)} validated B1 masks until {TARGET_COUNT} outputs reached...')

done, skipped, failed = [], [], []
aug_round = 0

while count_outputs() < TARGET_COUNT:
    print(f'\nRound {aug_round} - currently {count_outputs()}/{TARGET_COUNT} images')

    for i, mask_file in enumerate(valid_masks):
        if count_outputs() >= TARGET_COUNT:
            break

        basename = mask_file.replace('_mask.png', '')
        original_path = originals_dir / f'{basename}.JPG'
        mask_path = masks_dir / mask_file

        # Cycle prompts across rounds so repeated masks get different backgrounds
        category, prompt = ALL_PROMPTS[(i + aug_round * 7) % len(ALL_PROMPTS)]
        seed = 1000 + i + aug_round * 10000
        output_path = output_dir / f'{basename}_bg_{category}_r{aug_round}.jpg'

        if output_path.exists():
            skipped.append(basename)
            continue
        if not original_path.exists():
            failed.append(basename)
            continue

        try:
            t0 = time.time()
            inpaint_background(str(original_path), str(mask_path), str(output_path),
                               prompt=prompt, seed=seed)
            done.append(basename)
            print(f'[R{aug_round} {i+1}/{len(valid_masks)}] OK  {basename} '
                  f'({category})  {time.time() - t0:.1f}s  [total: {count_outputs()}]')
        except Exception as e:
            failed.append(basename)
            print(f'[R{aug_round} {i+1}/{len(valid_masks)}] FAIL {basename}: {e}')

        gc.collect()
        torch.cuda.empty_cache()

    aug_round += 1

print(f'\nComplete: {len(done)} generated this session, '
      f'{len(skipped)} skipped, {len(failed)} failed')
print(f'Total images in {output_dir}: {count_outputs()}')